In [18]:
import pandas as pd

datos = pd.read_csv("../data/processed/datos_procesados_2021.csv")

ventas = pd.to_numeric(
    datos["Venta$"].astype("string")
    .str.replace(r"[$,]", "", regex=True)
    .str.strip(),
    errors="coerce"
)
ventas_netas = pd.to_numeric(datos["VentasNetas"], errors="coerce")

filtro_ventas = (
    datos["TipoRegistro"].eq("Registro de venta")
    & ventas_netas.eq(1)
    & ventas.gt(0)
    & datos["Fuente"].isin(["Q1", "Q2"])
)

datos_ventas = datos.loc[filtro_ventas].copy()
datos_ventas["Vin"] = datos_ventas["Vin"].astype("string").str.strip()

print("Datos:", datos.shape)
print("Registros que cumplen los filtros:", len(datos_ventas))

Datos: (107787, 42)
Registros que cumplen los filtros: 93108



Pregunta 1 (3 pts): Retención de clientes
¿Cuántos VINs únicos visitaron el taller tanto en Q1 como en Q2? Lista los primeros 10 VINs y su número total de visitas en el semestre.

In [13]:
def mostrar_top10(df, nombre):
    resumen = (
        df.loc[df["Vin"].fillna("").ne("")]
        .groupby("Vin", as_index=False)
        .agg(Visitas=("NumeroOT", "nunique"))
        .sort_values(["Visitas", "Vin"], ascending=[False, True])
        .reset_index(drop=True)
    )
    print(f"\n{nombre} — VINs únicos: {len(resumen)}")
    display(resumen.head(10))

subconjuntos = {
    "Q1 + Q2": datos_ventas,
    "Q1": datos_ventas.loc[datos_ventas["Fuente"].eq("Q1")],
    "Q2": datos_ventas.loc[datos_ventas["Fuente"].eq("Q2")],
}

for nombre, subconjunto in subconjuntos.items():
    mostrar_top10(subconjunto, nombre)


Q1 + Q2 — VINs únicos: 68100


,Vin,Visitas
0,3KPA34AC8JE108967,11
1,2A4RR6DG9BR760969,9
2,3N1CN7AD2LK414311,8
3,5KBRL6889KB800670,8
4,1GKKR9KD4EJ125480,7
5,3G1J86CC1GS605484,7
6,3KPFK4A78JE149997,7
7,3N1CN7AD9LK411485,7
8,3N1CN8AE3LL906340,7
9,3TMAZ5CN6LM124144,7



Q1 — VINs únicos: 38030


,Vin,Visitas
0,3KPFK4A78JE149997,7
1,KMHC75LC0KU156042,6
2,KNDJP3A48K7671204,6
3,LSGHD52H2LD083455,6
4,1GTU99EL6KZ316547,5
5,2HGFC3277KH850558,5
6,3C6SRBDG5KG561656,5
7,3HGRM3850FG005153,5
8,3KPA34AC6JE020273,5
9,3KPC25B51KE042605,5



Q2 — VINs únicos: 37677


,Vin,Visitas
0,3KPA34AC8JE108967,10
1,3GNCJ7CE6JL261123,6
2,KL8CD6AD6FC777577,6
3,19UUB1654HA900038,5
4,1G1F91R78J0119978,5
5,2A4RR6DG9BR760969,5
6,3G1J85CC6GS591544,5
7,3G1J86CC1GS605484,5
8,3HGRU5893KM009711,5
9,3KPF14BD7LE185724,5


Pregunta 2 (5 pts): Distribución de órdenes
Calcula el porcentaje de órdenes de trabajo por Sucursal y por Tipo de Orden. Presenta el resultado en una tabla pivote que muestre: Sucursal | Tipo Orden | % del Total de Órdenes.

In [ ]:
datos_tabla = datos_ventas.assign(
    Sucursal=datos_ventas["Nombre"].fillna("Sin Definir"),
    TipoOrden=datos_ventas["TipoOrden"].fillna("Sin Definir")
)

ordenes = datos_tabla.drop_duplicates(
    subset=["Client", "Branch", "NumeroOT"]
)
total_ordenes = len(ordenes)

if total_ordenes == 0:
    raise ValueError("No hay órdenes que cumplan los filtros.")

tabla_pivote = ordenes.pivot_table(
    index=["Sucursal", "TipoOrden"],
    values="NumeroOT",
    aggfunc="count",
    fill_value=0
)
tabla_pivote["% del Total de Órdenes"] = (
    tabla_pivote["NumeroOT"] / total_ordenes * 100
).round(2)

tabla_pivote = (
    tabla_pivote[["% del Total de Órdenes"]]
    .reset_index()
    .rename(columns={"TipoOrden": "Tipo Orden"})
    .sort_values("% del Total de Órdenes", ascending=False)
    .reset_index(drop=True)
)

display(tabla_pivote)

,Sucursal,Tipo Orden,% del Total de Órdenes
0,Sucursal K,PUBLICO,6.53
1,Sucursal G,PUBLICO,5.30
2,Sucursal F,PUBLICO,5.07
3,Sucursal P,PUBLICO,4.65
4,Sucursal Q,PUBLICO,4.39
...,...,...,...
128,Sucursal N,Sin Definir,0.00
129,Sucursal R,Sin Definir,0.00
130,Sucursal M,Sin Definir,0.00
131,Sucursal U,ASEGURADORA,0.00


Pregunta 3 (4 pts): Segmentación de clientes VIP
Identifica los 5 VINs con mayor frecuencia de visitas en el semestre. Calcula el ticket promedio de este grupo y compáralo con el ticket promedio del resto de VINs. ¿Qué porcentaje representa la diferencia?

In [15]:
datos_vip = datos_ventas.loc[
    datos_ventas["Vin"].fillna("").ne("")
].copy()

datos_vip["ImporteVenta"] = ventas.loc[datos_vip.index]

ordenes = (
    datos_vip.groupby(
        ["Vin", "Client", "Branch", "NumeroOT"],
        as_index=False
    )
    .agg(Ticket=("ImporteVenta", "sum"))
)

resumen_vins = (
    ordenes.groupby("Vin", as_index=False)
    .agg(
        Visitas=("NumeroOT", "size"),
        TicketPromedio=("Ticket", "mean")
    )
    .sort_values(
        ["Visitas", "Vin"],
        ascending=[False, True]
    )
)

top5_vip = resumen_vins.head(5)
vins_vip = top5_vip["Vin"]

ordenes_vip = ordenes[ordenes["Vin"].isin(vins_vip)]
ordenes_resto = ordenes[~ordenes["Vin"].isin(vins_vip)]

ticket_vip = ordenes_vip["Ticket"].mean()
ticket_resto = ordenes_resto["Ticket"].mean()
diferencia_porcentual = (ticket_vip - ticket_resto) / ticket_resto * 100

print("Top 5 VINs VIP:")
display(top5_vip)

print(f"Ticket promedio del grupo VIP: ${ticket_vip:,.2f}")
print(f"Ticket promedio del resto:     ${ticket_resto:,.2f}")
print(f"Diferencia porcentual:          {diferencia_porcentual:+.2f}%")

Top 5 VINs VIP:


,Vin,Visitas,TicketPromedio
21145,3KPA34AC8JE108967,11,1170.365455
7985,2A4RR6DG9BR760969,9,3242.254444
27027,3N1CN7AD2LK414311,8,350.1375
31747,5KBRL6889KB800670,8,3294.61875
4231,1GKKR9KD4EJ125480,7,7652.258571


Ticket promedio del grupo VIP: $2,901.82
Ticket promedio del resto:     $2,145.29
Diferencia porcentual:          +35.26%


Pregunta 4 (4 pts): Análisis de cancelaciones
Identifica qué sucursales tienen el mayor porcentaje de cancelaciones (VentasNetas negativas) sobre el total de sus transacciones. Muestra el top 5 de sucursales con mayor tasa de cancelación.

In [19]:
base = datos.copy()
base["VentasNetas"] = pd.to_numeric(base["VentasNetas"], errors="coerce")

base = base.loc[
    base["VentasNetas"].notna() & base["VentasNetas"].ne(0)
].copy()

base["Sucursal"] = base["Nombre"].fillna("Sin Definir")
base["EsCancelacion"] = base["VentasNetas"].eq(-1)

id_transaccion = ["Fuente", "FechaFactura", "Client", "Branch", "Factura"]

transacciones = (
    base.groupby(id_transaccion, dropna=False)
    .agg(
        Sucursal=("Sucursal", "first"),
        EsCancelacion=("EsCancelacion", "any")
    )
    .reset_index()
)

resumen = (
    transacciones.groupby("Sucursal")
    .agg(
        TotalTransacciones=("EsCancelacion", "size"),
        Cancelaciones=("EsCancelacion", "sum")
    )
)

resumen["PorcentajeCancelacion"] = (
    resumen["Cancelaciones"] / resumen["TotalTransacciones"] * 100
).round(2)

top_5_sucursales = (
    resumen.sort_values("PorcentajeCancelacion", ascending=False)
    .head(5)
    .reset_index()
    .rename(columns={"PorcentajeCancelacion": "% de Cancelaciones"})
)

display(top_5_sucursales)

,Sucursal,TotalTransacciones,Cancelaciones,% de Cancelaciones
0,Sucursal V,4408,331,7.51
1,Sucursal U,947,50,5.28
2,Sucursal E,324,15,4.63
3,Sucursal D,2272,63,2.77
4,Sucursal B,1866,51,2.73


Pregunta 5 (4 pts): Tendencia mensual
Calcula las ventas netas totales por mes del semestre. ¿En qué mes se registró el mayor crecimiento porcentual respecto al mes anterior?


In [21]:
mensual = datos[
    ["Fuente", "FechaFactura", "Client", "Branch", "Factura", "TipoRegistro", "Venta$"]
].copy()

mensual["FechaFactura"] = pd.to_datetime(
    mensual["FechaFactura"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

mensual["Venta$"] = pd.to_numeric(
    mensual["Venta$"].astype("string")
    .str.replace(r"[$,]", "", regex=True)
    .str.strip(),
    errors="coerce"
)


mensual = mensual.loc[
    mensual["Fuente"].isin(["Q1", "Q2"])
    & mensual["TipoRegistro"].eq("Registro de venta")
    & mensual["Venta$"].gt(0)
    & mensual["FechaFactura"].notna()
].copy()

mensual = mensual.drop_duplicates(
    subset=["Fuente", "FechaFactura", "Client", "Branch", "Factura"]
)

meses = pd.period_range("2021-01", "2021-06", freq="M")
ventas_por_mes = (
    mensual.groupby(mensual["FechaFactura"].dt.to_period("M"))
    .size()
    .reindex(meses, fill_value=0)
)

mes_anterior = ventas_por_mes.shift(1)
crecimiento = (
    (ventas_por_mes - mes_anterior) / mes_anterior * 100
).where(mes_anterior.ne(0)).round(2)

nombres_meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo",
    4: "Abril", 5: "Mayo", 6: "Junio"
}

resultado = pd.DataFrame({
    "Mes": [f"{nombres_meses[mes.month]} {mes.year}" for mes in meses],
    "Cantidad de ventas": ventas_por_mes.to_numpy(),
    "Crecimiento respecto al mes anterior (%)": crecimiento.to_numpy()
})

display(resultado)

crecimiento_valido = crecimiento.dropna()
if not crecimiento_valido.empty:
    mes_maximo = crecimiento_valido.idxmax()
    print(
        f"Mayor crecimiento: {nombres_meses[mes_maximo.month]} "
        f"({crecimiento_valido.max():.2f}%)"
    )
else:
    print("No se pudo calcular el crecimiento mensual.")

,Mes,Cantidad de ventas,Crecimiento respecto al mes anterior (%)
0,Enero 2021,14777,NaN
1,Febrero 2021,14590,-1.27
2,Marzo 2021,17563,20.38
3,Abril 2021,14967,-14.78
4,Mayo 2021,15299,2.22
5,Junio 2021,15911,4.00


Mayor crecimiento: Marzo (20.38%)
